In [3]:
import sys
import gzip
from datetime import datetime
from alignment.PoseGraph import PoseGraph

import google.protobuf
import delimited_protobuf
from protocol import message_formats_pb2, telemetry_pb2, control_pb2, req_rep_pb2
import numpy as np

In [2]:
with (gzip.open("../logs/19_04_2024/BYEDP210004_ee6c4d5b1b0969d4_00514.bez", "rb") as bez,
      gzip.open("../logs/19_04_2024/multibeam_BYEDP210004_2024-04-19_132753.995.bez", "rb") as mbez):
    while True:
        try:
            binlog = delimited_protobuf.read(bez, message_formats_pb2.BinlogRecord)
            if telemetry_pb2.ImuTel.DESCRIPTOR.full_name not in binlog.payload.type_url:
                continue
            telemetry = telemetry_pb2.ImuTel()
            telemetry.ParseFromString(binlog.payload.value)

            binlog.clock_monotonic.seconds

            print(telemetry)
            # ping = telemetry_pb2.OculusPingTel()
            # ping.ParseFromString(binlog.payload.value)
            # print(ping.ping, binlog.clock_monotonic.seconds + binlog.clock_monotonic.nanos / 1e9)
        except Exception as e:
            print(e)
            break

module 'protocol.telemetry_pb2' has no attribute 'ImuTel'


In [4]:
def read_binlog(file_path, type_names=[telemetry_pb2.MultibeamPingTel]):
    with gzip.open(file_path, "rb") as fin:
        while True:
            binlog = delimited_protobuf.read(fin, message_formats_pb2.BinlogRecord)
            if not binlog:
                break
            type_name = next((type_name for type_name in type_names if
                              type_name.DESCRIPTOR.full_name in binlog.payload.type_url), None)
            if not type_name:
                continue
            message = type_name()
            message.ParseFromString(binlog.payload.value)
            yield message, binlog.clock_monotonic.seconds + binlog.clock_monotonic.nanos / 1e9, binlog.unix_timestamp.seconds + binlog.unix_timestamp.nanos / 1e9, 


# All messages will be yielded in chronological order
def read_combined(file_paths, type_names=[telemetry_pb2.MultibeamPingTel]):
    generators = [read_binlog(file_path, type_names) for file_path in file_paths]
    buffer = []
    remove = []

    start_ts = None
    interval = 5
    
    if telemetry_pb2.MultibeamPingTel in type_names:
        iteration = 0
        while not start_ts and generators and iteration < 5:
            for i, generator in enumerate(generators):
                try:
                    message, ts, uts = next(generator)
                except:
                    remove.append(i)
                    break
                    
                buffer.append((message, ts, uts))
                
                if isinstance(message, telemetry_pb2.MultibeamPingTel):
                    start_ts = ts
                    break
            
            for i in remove:
                generators.pop(i)
            remove = []
            
            iteration += 1
                    
        if start_ts:
            buffer = [x for x in buffer if x[1] >= start_ts]
    
    while generators:
        for i, generator in enumerate(generators):
            ts = start_ts
            while ts - start_ts < interval:
                try:
                    message, ts, uts = next(generator)
                    if ts > start_ts:
                        buffer.append((message, ts, uts))
                except StopIteration:
                    remove.append(i)
                    break
        
        
        for i in remove:
            generators.pop(i)
        remove = []
        
        
        buffer.sort(key=lambda x: x[1])

        for message, ts, uts in buffer:
            if start_ts <= ts < start_ts + interval:
                yield message, ts, uts
                
        start_ts += interval


In [ ]:
i = 0
for message, ts, uts in read_combined(
        # ["../logs/BYEDP210001_ee6c4d5b1c0c71d4_01077.bez", "../logs/multibeam_BYEDP210001_2024-07-19_130711.mbez"],
        ["../logs/multibeam_BYEDP210001_2024-07-19_130711.mbez"],
        # [req_rep_pb2.SyncTimeReq, telemetry_pb2.PositionEstimateTel, telemetry_pb2.MultibeamPingTel, telemetry_pb2.CalibratedImuTel]):
        [telemetry_pb2.MultibeamPingTel]):
    print(message)
    break
    if i > 10:
        break
    i += 1
    
    
    
#     
#     
# for message, ts in read_combined(
#         ["../logs/multibeam_BYEDP210001_2024-07-19_125218.mbez"],
#         [telemetry_pb2.MultibeamPingTel]):
#     print(type(message), ts)
#     break


ping {
  range: 30.758804321289062
  gain: 100.0
  frequency: 1196808.5106382978
  speed_of_sound_used: 1517.4982581645493
  frequency_mode: MULTIBEAM_FREQUENCY_MODE_LOW_FREQUENCY
  number_of_ranges: 378
  number_of_beams: 512
  step: 512
  bearings: -65.0
  bearings: -64.5199966430664
  bearings: -64.05000305175781
  bearings: -63.59000015258789
  bearings: -63.13999938964844
  bearings: -62.689998626708984
  bearings: -62.25
  bearings: -61.81999969482422
  bearings: -61.38999938964844
  bearings: -60.970001220703125
  bearings: -60.54999923706055
  bearings: -60.13999938964844
  bearings: -59.72999954223633
  bearings: -59.33000183105469
  bearings: -58.939998626708984
  bearings: -58.54999923706055
  bearings: -58.15999984741211
  bearings: -57.77000045776367
  bearings: -57.400001525878906
  bearings: -57.02000045776367
  bearings: -56.650001525878906
  bearings: -56.279998779296875
  bearings: -55.91999816894531
  bearings: -55.560001373291016
  bearings: -55.20000076293945
  bea

In [15]:
import g2o
from alignment.plot_slam2d import plot_slam2d

generator = read_binlog("../logs/BYEDP210001_ee6c4d5b1c0c71d4_01077.bez",
                        [req_rep_pb2.SyncTimeReq, telemetry_pb2.PositionEstimateTel])

time_offset = 0
time_offset_monotonic = 0

pose_graph = PoseGraph(verbose=True)

# pose_graph.add_fixed_pose(g2o.SE2())

count = 0
last_time = 0
freq = 0
for message, monotonic, unix in generator:
    # Correct the timestamps
    # monotonic = unix
    unix += time_offset

    if isinstance(message, req_rep_pb2.SyncTimeReq):
        time_offset = message.time.unix_timestamp.seconds
        time_offset_monotonic = monotonic
        # print("Time offset:", time_offset)

        # print(datetime.fromtimestamp(time_offset))
        continue

    if time_offset == 0:
        continue

    if isinstance(message, telemetry_pb2.PositionEstimateTel):
        if not message.position_estimate.is_valid:
            print(message)
            continue
        print(f"Valid position estimate {monotonic}")

    if count % 20 == 0:
        freq = 20 / (unix - last_time)
        last_time = unix
        print(f"Frequency: {freq} Hz")
        print(message)
    # pose_graph.add_odometry(
    #         message.position_estimate.northing,
    #         message.position_estimate.easting,
    #         message.position_estimate.heading,
    #         np.eye(3))
    count += 1
    if count > 2000:
        break



# fig = plot_slam2d(pose_graph.optimizer, "Before optimisation")
# fig.write_image("before_optimisation.png")
# fig.write_html("before_optimisation.html")
# fig.show("notebook")



position_estimate {
  heading: -3.962458947626146e-07
  yaw_rate: 1.8928521967609413e-05
  global_position {
  }
  navigation_sensors {
    sensor_id: NAVIGATION_SENSOR_ID_WATERLINKED_DVL_A50
  }
}

position_estimate {
  heading: -1.2346617950242944e-08
  yaw_rate: 9.695063454273622e-06
  global_position {
  }
  navigation_sensors {
    sensor_id: NAVIGATION_SENSOR_ID_WATERLINKED_DVL_A50
  }
}

position_estimate {
  heading: -6.306887012641482e-09
  yaw_rate: 4.9657473937259056e-06
  global_position {
  }
  navigation_sensors {
    sensor_id: NAVIGATION_SENSOR_ID_WATERLINKED_DVL_A50
  }
}

position_estimate {
  northing: -2.633068620384589e-16
  easting: 1.4998970121471444e-10
  heading: 2.427341598831845e-07
  surge_rate: 1.3221492280721722e-14
  sway_rate: -3.5141300031682476e-05
  yaw_rate: -1.6996675185509957e-05
  ocean_current: 4.305842399597168
  global_position {
  }
  navigation_sensors {
    sensor_id: NAVIGATION_SENSOR_ID_WATERLINKED_DVL_A50
  }
}

position_estimate {
  nort

EOFError: Compressed file ended before the end-of-stream marker was reached

In [39]:
generator = read_binlog("../logs/19_04_2024/multibeam_BYEDP210004_2024-04-19_132753.995.bez",
                        [telemetry_pb2.OculusPingTel])

time_offset = 0
time_offset_monotonic = 0

pose_graph = PoseGraph(verbose=True)

pose_graph.add_fixed_pose(g2o.SE2())

count = 0
for message, monotonic, unix in generator:
    print(message)
    break
